<a href="https://colab.research.google.com/github/MajidSharaf/CampaignLens/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
from textblob import TextBlob
import requests

In [3]:
!pip install emoji langdetect
import emoji
from langdetect import detect_langs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 34.1 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=840d96e5a31581f1516fb358f3dd5d8f89302bc58b962e4dfec4f5070d9a1aff
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [4]:
import re
import string
!pip install contractions
import contractions
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

In [6]:
import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')
nltk.download('gutenberg')
nltk.download('wordnet')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
from nltk.stem import WordNetLemmatizer


In [8]:
url = "https://raw.githubusercontent.com/rishabhverma17/sms_slang_translator/master/slang.txt"
response = requests.get(url)

slang = {}
for line in response.text.splitlines():
    if '=' in line:
        key, value = line.split('=', 1)
        slang[key.strip().lower()] = value.strip().lower()

In [9]:
!wget "https://raw.githubusercontent.com/MajidSharaf/CampaignLens/main/Datasets/Versions/comments.csv"
Trump = pd.read_csv("/content/comments.csv")

--2026-06-12 21:04:30--  https://raw.githubusercontent.com/MajidSharaf/CampaignLens/main/Datasets/Versions/comments.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4511554 (4.3M) [text/plain]
Saving to: ‘comments.csv’

comments.csv        100%[===================>]   4.30M  --.-KB/s    in 0.08s   

2026-06-12 21:04:30 (56.2 MB/s) - ‘comments.csv’ saved [4511554/4511554]



#Data Exploratory

In [10]:
print(len(Trump))
Trump.head(5)

25214


,author,updated_at,like_count,text,video_id,public
0,@shivagarwal564,2026-06-01T12:33:29Z,0,I don't want all this news only google,qLhJuLAnY4M,True
1,@mechniack,2026-06-01T12:33:20Z,0,Trump the dumbest President in American histor...,qLhJuLAnY4M,True
2,@MCMTL,2026-06-01T12:23:39Z,1,Fareed knows full well that the reason many ob...,qLhJuLAnY4M,True
3,@prajnasamadhi60,2026-06-01T12:18:13Z,0,China's Xinjiang genocide? Millions visited Xi...,qLhJuLAnY4M,True
4,@ftgujyty,2026-06-01T12:11:15Z,0,jet pilots need precision,qLhJuLAnY4M,True


In [11]:
Trump.columns.tolist()

['author', 'updated_at', 'like_count', 'text', 'video_id', 'public']

In [12]:
Trump = Trump[['text', 'like_count']]

In [13]:
Trump.dropna()

,text,like_count
0,I don't want all this news only google,0
1,Trump the dumbest President in American histor...,0
2,Fareed knows full well that the reason many ob...,1
3,China's Xinjiang genocide? Millions visited Xi...,0
4,jet pilots need precision,0
...,...,...
25209,It amazes me that so many people failed to rea...,411
25210,Dude is stupid 😂,41
25211,You can't buy this level of class,1602
25212,Trump is the king of inappropriate jokes,126


#PreProcessing

In [14]:
class RemoveURLs(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [re.sub(r'http\S+|www\S+', '', str(text)) for text in X]

In [15]:
class FixContractions(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [contractions.fix(text) for text in X]

In [16]:
class FixRepeatedChars(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [re.sub(r'(.)\1{2,}', r'\1\1', text) for text in X]

In [17]:
class FixSpelling(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [str(TextBlob(text).correct()) for text in X]

In [18]:
class Tokenize(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [word_tokenize(text.lower()) for text in X]

In [19]:
class RemovePunctAndNumbers(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [[w for w in tokens if w not in string.punctuation and not w.isdigit()] for tokens in X]

In [20]:
class FixSlang(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        result = []
        for tokens in X:
            tokens = [slang.get(w, w) for w in tokens]
            tokens = [w for phrase in tokens for w in phrase.split()]
            result.append(tokens)
        return result

In [21]:
class HandleNegations(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        result = []
        for tokens in X:
            handled = []
            i = 0
            while i < len(tokens):
                if tokens[i] == 'not' and i + 1 < len(tokens):
                    handled.append(f"not_{tokens[i+1]}")
                    i += 2
                else:
                    handled.append(tokens[i])
                    i += 1
            result.append(handled)
        return result

In [22]:
class RemoveStopwords(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        stop_words = set(stopwords.words('english'))
        stop_words.discard('not')
        stop_words.discard('no')
        stop_words.discard('nor')
        return [[w for w in tokens if w not in stop_words] for tokens in X]

In [23]:
class Lemmatize(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        lemmatizer = WordNetLemmatizer()
        return [[lemmatizer.lemmatize(w) for w in tokens] for tokens in X]

In [24]:
class JoinTokens(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [' '.join(tokens) for tokens in X]

In [25]:
class DropNA(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return pd.Series(X).dropna().reset_index(drop=True).tolist()

In [26]:
class ConvertEmojis(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [emoji.demojize(text, delimiters=(' ', ' ')) for text in X]

In [27]:
class FixHashtagsAndMentions(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        result = []
        for text in X:
            text = re.sub(r'@\w+', '', text)           # drop mentions
            text = re.sub(r'#(\w+)', r'\1', text)      # hashtag → word
            result.append(text)
        return result

In [28]:
class KeepEnglishOnly(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = list(X)
        result = []
        for text in X:
            if not isinstance(text, str) or not text.strip():
                continue
            ascii_ratio = sum(c.isascii() for c in text) / len(text)
            if ascii_ratio >= 0.85:
                result.append(text)
        return result

In [29]:
class MinLengthFilter(BaseEstimator, TransformerMixin):
    def __init__(self, min_words=3):
        self.min_words = min_words
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [text for text in X if len(text.split()) >= self.min_words]

In [30]:
text_pipeline = Pipeline([
    ('drop_na',              DropNA()),
    ('convert_emojis',       ConvertEmojis()),
    #('keep_english',         KeepEnglishOnly()),
    ('fix_hashtags',         FixHashtagsAndMentions()),
    ('remove_urls',          RemoveURLs()),
    ('fix_contractions',     FixContractions()),
    ('fix_repeated',         FixRepeatedChars()),
    #('fix_spelling',         FixSpelling()),
    ('tokenize',             Tokenize()),
    ('remove_punct',         RemovePunctAndNumbers()),
    ('fix_slang',            FixSlang()),
    ('handle_negations',     HandleNegations()),
    ('remove_stopwords',     RemoveStopwords()),
    ('lemmatize',            Lemmatize()),
    ('join_tokens',          JoinTokens()),
    ('min_length',           MinLengthFilter(min_words=3)),
])

In [31]:
Trump_Processed = pd.Series(text_pipeline.fit_transform(Trump['text']))

In [32]:
Trump_Processed.head()

,0
0,not_want news google
1,trump dumbest president american history terri...
2,fareed know full well reason many objected jcp...
3,china 's xinjiang genocide million visited xin...
4,jet pilot need precision


In [33]:
# save cleaned comments for next pipeline steps
Trump_Processed_df = pd.DataFrame({'cleaned_text': Trump_Processed})
Trump_Processed_df.to_csv('comments_cleaned.csv', index=False)
print("Saved! Shape:", Trump_Processed_df.shape)
print(Trump_Processed_df.head())

Saved! Shape: (22174, 1)
                                        cleaned_text
0                               not_want news google
1  trump dumbest president american history terri...
2  fareed know full well reason many objected jcp...
3  china 's xinjiang genocide million visited xin...
4                           jet pilot need precision


In [34]:
print(len(Trump_Processed))

22174
